<div align="center">

# 🚕 Yandex Go Analytics

## 6 · Data Enrichment

*Join every trip with its driver's profile in a single, analysis-ready table.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![operation](https://img.shields.io/badge/operation-LEFT%20JOIN-00ADD4)
![table](https://img.shields.io/badge/table-enriched_trips-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▰▱▱  **6 / 8**

---

## 🎯 Task requirements

- [x] Generate a **synthetic `driver_id`** for the taxi trips
- [x] Perform a **`LEFT JOIN`** between the trips and the driver profiles
- [x] Save the unified, enriched dataset as **`gold.enriched_trips`**

## 🧩 How it is solved

**Synthetic key.** The trip generator assigns every trip a `driver_id` drawn from a *deterministic* UUID pool
(fixed Faker seed). The driver generator uses the **same** pool, so trips and driver profiles share a join key
without any real-world identifiers.

```text
 silver.taxi  (stream, de-duplicated by trip id)
      │  driver_id
      ▼
   LEFT JOIN  ◀──────────  silver.drivers  (static snapshot, one row per driver id)
      │
      ▼
 gold.enriched_trips   =   all trip columns  +  driver_name · driver_car_number
                                                driver_experience · driver_rating [· driver_phone]
```

| Design choice | Why |
|:--|:--|
| `LEFT JOIN` | A trip must survive even when its driver profile is missing or was rejected by Silver |
| Stream ⨝ static | New trips flow in incrementally; the (small) driver dimension is read as a snapshot |
| `driver_` prefix | Keeps trip and driver columns unambiguous |
| `driver_phone` only if present | The `phone` column was rolled back in notebook 3, so it is added only if it exists in Silver |

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-enrichment")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 12:56:40.530 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-enrichment | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 2️⃣ Build `gold.enriched_trips`

In [2]:
from src.core.etl.gold import EnrichedTripsGoldService

EnrichedTripsGoldService().run(spark)

enriched = spark.table(f"{CATALOG}.gold.enriched_trips")
print(f"gold.enriched_trips: {enriched.count():,} rows | {len(enriched.columns)} columns")
display(enriched)

2026-10-09 12:56:40.540 | INFO     | src.core.etl.gold.trip_driver:run:34 - Reading stream from silver.taxi via checkpoints...
2026-10-09 12:57:11.146 | INFO     | src.core.etl.gold.trip_driver:run:73 - Enriched yandex_go_trips_pipeline successfully written!


gold.enriched_trips: 10,000 rows | 32 columns


DataFrame[driver_id: string, id: string, user_id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: bigint, trip_distance: double, rate_code_id: bigint, store_and_fwd_flag: string, pu_location_id: int, do_location_id: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, _rescued_data: string, _ingested_at: timestamp, _source_file: string, payment_type_description: string, _processed_at: timestamp, driver_name: string, driver_car_number: string, driver_experience: int, driver_rating: double, _calculated_at: timestamp]

## 3️⃣ Verify the join

In [3]:
def check(label: str, condition: bool) -> None:
    print(("✅" if condition else "❌"), label)
    assert condition, label

silver_trip_count = spark.table(f"{CATALOG}.silver.taxi").select("id").distinct().count()
enriched_count = enriched.count()

check(
    f"row-preserving: {enriched_count:,} enriched rows == {silver_trip_count:,} distinct Silver trips",
    enriched_count == silver_trip_count,
)
check("one row per trip id", enriched_count == enriched.select("id").distinct().count())
check(
    "driver columns are present",
    {"driver_name", "driver_car_number", "driver_experience", "driver_rating"} <= set(enriched.columns),
)

✅ row-preserving: 10,000 enriched rows == 10,000 distinct Silver trips
✅ one row per trip id
✅ driver columns are present


### 🎯 Match rate

Trips whose driver has no Silver profile keep `NULL` driver columns — the signature of a `LEFT JOIN`.

In [4]:
match = (
    enriched.agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("driver_name").isNotNull(), 1).otherwise(0)).alias("with_driver"),
        F.sum(F.when(F.col("driver_name").isNull(), 1).otherwise(0)).alias("without_driver"),
    )
    .withColumn("match_rate_pct", F.round(F.col("with_driver") / F.col("total_trips") * 100, 1))
)
display(match)

assert match.first()["with_driver"] > 0, "No trip was matched with a driver profile"
print("✅ Trips are enriched with driver profiles")

DataFrame[total_trips: bigint, with_driver: bigint, without_driver: bigint, match_rate_pct: double]

✅ Trips are enriched with driver profiles


### 🏆 What the enriched table unlocks

Revenue per driver — impossible without the join.

In [5]:
top_drivers = (
    enriched.filter(F.col("driver_name").isNotNull())
    .groupBy("driver_id", "driver_name", "driver_experience", "driver_rating")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("fare_amount"), 2).alias("revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.col("revenue").desc())
)
display(top_drivers)

DataFrame[driver_id: string, driver_name: string, driver_experience: int, driver_rating: double, trips: bigint, revenue: double, avg_distance: double]

---

## ✅ Checkpoint

- [x] Synthetic `driver_id` shared by trips and driver profiles
- [x] `LEFT JOIN` of `silver.taxi` with `silver.drivers`
- [x] `gold.enriched_trips` written; no trip lost, match rate reported

> **Next up ▶ [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb)** — recreate the whole pipeline declaratively with Delta Live Tables.

<div align="center"><sub>Yandex Go Analytics · notebook 6 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>